# 03 — Exploratory Data Analysis (Karnataka Infrastructure & EV Adoption)

**Key Questions Explored:**
1. How are ChargeWise's 86 Karnataka stations distributed across districts and operators?
2. What proportion of listed chargers support true DC fast charging (`>= 30 kW`) vs AC / 2-wheeler swap vs unknown power?
3. How do 2-Wheeler and 4-Wheeler EV registrations in Karnataka evolve from FY2022 to FY2024?
4. Which EV manufacturers lead 2W and 4W adoption in India?

In [1]:
from pathlib import Path
import pandas as pd

REPO = Path("..").resolve()
PROC = REPO / "data" / "processed"
RAW = REPO / "data" / "raw" / "india_ev_market"

stations = pd.read_csv(PROC / "karnataka_stations_clean.csv")
ka_sales = pd.read_csv(PROC / "karnataka_ev_sales_clean.csv", parse_dates=["date_parsed"])
makers = pd.read_csv(RAW / "electric_vehicle_sales_by_makers.csv")

print("=== Station Counts by District ===")
print(stations["district"].value_counts())

print("\n=== Station Counts by Operator ===")
print(stations["operator"].value_counts().head(12))

print("\n=== Charging Type Breakdown ===")
print(stations["charging_type"].value_counts())

print("\n=== Verification Status Breakdown ===")
print(stations["verification_status"].value_counts())

=== Station Counts by District ===
district
Bengaluru Urban     55
Karnataka            9
Dakshina Kannada     5
Mysuru               4
Kodagu               4
Hassan               4
North Karnataka      4
Belagavi             1
Name: count, dtype: int64

=== Station Counts by Operator ===
operator
Unlisted / Independent           59
Zeon Charging                     7
Tata Power EZ Charge              6
Statiq                            4
BESCOM EV Mithra                  3
Ather Grid                        2
Shell Recharge                    2
KPTCL                             1
Glida (formerly Fortum India)     1
Jio-bp pulse                      1
Name: count, dtype: int64

=== Charging Type Breakdown ===
charging_type
DC_FAST       66
SWAP_OR_2W    16
AC             2
MIXED          2
Name: count, dtype: int64

=== Verification Status Breakdown ===
verification_status
listed                         71
operational_status_verified    12
location_verified               3
Name: count, 

In [2]:
print("=== Rated Max Power (kW) Summary (excluding unknown power) ===")
print(stations["max_power_kw"].describe().round(2))

print("\n=== Karnataka EV Sales Summary by Vehicle Category (Apr 2021 - Mar 2024) ===")
ka_cat = ka_sales.groupby("vehicle_category").agg(
    months=("date_parsed", "count"),
    total_evs_sold=("electric_vehicles_sold", "sum"),
    total_vehicles_sold=("total_vehicles_sold", "sum"),
    mean_monthly_penetration_pct=("ev_penetration_pct", "mean"),
    peak_monthly_penetration_pct=("ev_penetration_pct", "max"),
)
ka_cat["overall_penetration_pct"] = (ka_cat["total_evs_sold"] / ka_cat["total_vehicles_sold"]) * 100
display(ka_cat.round(2))

print("\n=== Top 4-Wheeler & 2-Wheeler EV Makers (National VAHAN Dataset) ===")
display(
    makers.groupby(["vehicle_category", "maker"])["electric_vehicles_sold"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
    .reset_index()
)

=== Rated Max Power (kW) Summary (excluding unknown power) ===
count     17.00
mean      47.67
std       31.27
min        3.00
25%       25.00
50%       60.00
75%       60.00
max      120.00
Name: max_power_kw, dtype: float64

=== Karnataka EV Sales Summary by Vehicle Category (Apr 2021 - Mar 2024) ===


,months,total_evs_sold,total_vehicles_sold,mean_monthly_penetration_pct,peak_monthly_penetration_pct,overall_penetration_pct
vehicle_category,,,,,,
2-Wheelers,36,292329,3203736,8.47,18.37,9.12
4-Wheelers,36,20666,790593,2.39,5.54,2.61



=== Top 4-Wheeler & 2-Wheeler EV Makers (National VAHAN Dataset) ===


,vehicle_category,maker,electric_vehicles_sold
0,2-Wheelers,OLA ELECTRIC,489473
1,2-Wheelers,TVS,272575
2,2-Wheelers,ATHER,204449
3,2-Wheelers,HERO ELECTRIC,170394
4,2-Wheelers,AMPERE,167274
5,2-Wheelers,OKINAWA,165231
6,2-Wheelers,OTHERS,156471
7,2-Wheelers,BAJAJ,145606
8,4-Wheelers,Tata Motors,88935
9,4-Wheelers,Mahindra & Mahindra,41193
